# 15. Contextual Compression

Shrink each retrieved chunk to only the sentences that matter for the question.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Retrieved chunks contain a lot of text the question doesn't need. That wastes tokens and distracts the model (the 'lost in the middle' problem). **Contextual compression** runs an extra step that keeps only the sentences relevant to the question before generation.

## How it works

1. Retrieve a generous top-k (say 5)
2. For each chunk ask the LLM to copy only the useful sentences
3. Discard chunks that return NONE
4. Generate from the compressed context
5. Compare context size and answer quality with the uncompressed version

```text
 question --> retrieve k=5 chunks (2,500 chars)
          --> [compress each] --> relevant sentences only (600 chars)
          --> LLM --> answer
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
def compress(query, chunk):
    return llm(
        "Extract only the sentences from the passage that help answer the question. "
        "Copy them verbatim. If none are relevant reply with NONE.\n\n"
        f"Question: {query}\n\nPassage: {chunk}"
    )

query = "How is mean reciprocal rank calculated?"
hits = store.search(query, 5)
compressed = [c for c in (compress(query, h["text"]) for h in hits) if c.upper() != "NONE"]

before = sum(len(h["text"]) for h in hits)
after = sum(len(c) for c in compressed)
print(f"context shrank from {before} to {after} characters")

In [ ]:
print(answer(query, compressed))

Compression saves tokens and removes distractors, at the cost of one extra LLM call per chunk. Use a small, fast model for it.

## Look at one chunk before and after

In [ ]:
q = "What is the Reciprocal Rank Fusion constant?"
h = store.search(q, 1)[0]
print("BEFORE (%d chars):\n%s\n" % (len(h["text"]), h["text"]))
c = compress(q, h["text"])
print("AFTER (%d chars):\n%s" % (len(c), c))

## Quality and cost across golden questions
Compression must *shrink* the context without losing the evidence. We check both: size reduction and evidence still present.

In [ ]:
import pandas as pd
rows = []
for item in load_eval_questions()[:6]:
    hits = store.search(item["question"], 4)
    comp = [c for c in (compress(item["question"], h["text"]) for h in hits) if c.upper().strip() != "NONE"]
    before = sum(len(h["text"]) for h in hits); after = sum(len(c) for c in comp)
    rows.append({"question": item["question"][:45], "chars_before": before, "chars_after": after,
                 "reduction": f"{1 - after / before:.0%}",
                 "evidence_kept": any(item["evidence"].lower() in c.lower() for c in comp)})
res = pd.DataFrame(rows); display(res)
print("evidence still present after compression:", res["evidence_kept"].mean())

## Do answers stay as good?

In [ ]:
compare_pipelines({
    "full chunks": lambda q: answer(q, [h["text"] for h in store.search(q, 4)]),
    "compressed": lambda q: answer(q, [c for c in (compress(q, h["text"]) for h in store.search(q, 4)) if c.upper().strip() != "NONE"]),
}, n=5)

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Fewer tokens, lower cost | Extra LLM call per chunk |
| Removes distractors | Compressor can delete needed information |
| Lets you retrieve a larger k safely | Paraphrasing risk if not told to copy verbatim |

## When to use it

Large chunks, expensive generator models, or long-context prompts where noise hurts.

## Experiments to try

- Use embedding-based sentence filtering instead of an LLM and compare speed.
- Compress all chunks in a single prompt.
- Measure compression ratio vs answer accuracy at k=8.

## Key takeaways

- Extract, don't rewrite.
- Verify evidence survives compression.
- Use a small, fast model for the compressor.

---
**Next:** [16. Document Augmentation](../16_document_augmentation/document_augmentation.ipynb)